<a href="https://colab.research.google.com/github/ArchitPandey/RAG_Pipeline/blob/main/rag_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pymupdf4llm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.7/164.7 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 74.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.6/41.6 MB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 82.5 MB/s eta 0:00:00


In [2]:
import pymupdf4llm
import re

markdown_txt = pymupdf4llm.to_markdown('Introduction to Agents.pdf', header=False, footer=False, ignore_images=True, write_images=False)
#print(markdown_txt)

=== Document parser messages ===
Using Tesseract for OCR processing.
OCR on page.number=0/1.
OCR on page.number=1/2.
OCR on page.number=2/3.
OCR on page.number=3/4.
OCR on page.number=4/5.
OCR on page.number=11/12.
OCR on page.number=13/14.
OCR on page.number=24/25.
OCR on page.number=26/27.
OCR on page.number=27/28.
OCR on page.number=38/39.
OCR on page.number=44/45.
OCR on page.number=46/47.
OCR on page.number=47/48.
OCR on page.number=48/49.
OCR on page.number=49/50.



In [3]:
# clean md - remove image tags
markdown_txt = re.sub(
    r'<!-- Start of picture text -->.*?<!-- End of picture text -->',
    '',
    markdown_txt,
    flags=re.DOTALL
)

In [11]:
import pandas as pd

# chunking - trying recursive chunking since structural chunking cause chunk size to very big

chunk_config = [
    {
        "level": 1,
        "pattern": r"\n(?=##\s*\*\*)"
    },
    {
        "level": 2,
        "pattern": r"\n(?=###\s*\*\*)"
    },
    {
        "level": 3,
        "pattern": r"\n(?=####\s*\*\*)"
    }

]

def rec_chunker(txt, max_token, c_lvl):
  #print(f"txtlen = {len(txt)} c_lvl = {c_lvl}")
  if ((len(txt)//4) <= max_token):
    return [txt]
  if (c_lvl > len(chunk_config)):
    return [txt]

  intermediate_chunks = re.split(chunk_config[c_lvl-1]["pattern"], txt)
  final_chunks = []
  for ic in intermediate_chunks:
    if (ic.strip()):
      final_chunks.extend(rec_chunker(ic.strip(), max_token, c_lvl + 1))
  #print(f"txtlen = {len(txt)} c_lvl = {c_lvl} final_chunks {len(final_chunks)}")
  return final_chunks


chunks = rec_chunker(markdown_txt, 400, 1)

chunks_dict: list[dict] = []
for i, ch in enumerate(chunks):
  dict_element = {
      "num": i+1,
      "content": ch,
      "token": len(ch) //4
  }
  chunks_dict.append(dict_element)

df = pd.DataFrame(chunks_dict)
df

,num,content,token
0,1,# Google a \n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\...,27
1,2,## **From Predictive AI to Autonomous Agents**...,689
2,3,## **Introduction to AI Agents** \n\nIn the si...,1228
3,4,### **The Agentic Problem-Solving Process** \n...,950
4,5,### **Level 0: The Core Reasoning System** \n\...,253
5,6,### **Level 1: The Connected Problem-Solver** ...,234
6,7,### **Level 2: The Strategic Problem-Solver** ...,439
7,8,### **Level 3: The Collaborative Multi-Agent S...,308
8,9,### **Level 4: The Self-Evolving System** \n\n...,305
9,10,"## **Core Agent Architecture: Model, Tools, an...",64


In [33]:
df.describe()

,num,token
count,42.000000,42.000000
mean,21.500000,375.714286
std,12.267844,264.128691
min,1.000000,27.000000
25%,11.250000,190.250000
50%,21.500000,301.500000
75%,31.750000,486.500000
max,42.000000,1228.000000


In [34]:
def fixed_len_chunker(txt, chunk_len, overlap):
  idx: int = 0
  c = []
  while ( (idx+chunk_len) < len(txt)):
    #print(f"idx = {idx} chunk_len = {chunk_len}")
    c.append( txt[idx: idx+chunk_len])
    idx += chunk_len - overlap
  c.append(txt[idx:])
  return c

def child_chunk_obj(chunk_list, parent_num, child_chunk_st_idx):
  child_chunks_list = []
  ctr: int = 0
  for ch in chunk_list:
    dict_element = {
        "parent_num": parent_num,
        "child_num": child_chunk_st_idx + ctr,
        "content": ch,
        "token": len(ch) //4
    }
    child_chunks_list.append(dict_element)
    ctr +=1
  return child_chunks_list

def generate_child_chunks(parent_chunks_dict):
  child_chunks_dict: list[dict] = []
  child_chunk_idx: int = 0
  for parent_chunk in parent_chunks_dict:
    if parent_chunk["token"] > 384:
      fixed_len_chunks = fixed_len_chunker(parent_chunk["content"], 384*4, 50)
      child_chunks_dict.extend( child_chunk_obj ( fixed_len_chunks, parent_chunk["num"], child_chunk_idx ) )
      child_chunk_idx += len(fixed_len_chunks)
    else:
      child_chunks_dict.extend( child_chunk_obj([parent_chunk["content"]], parent_chunk["num"], child_chunk_idx)  )
      child_chunk_idx += 1
  return child_chunks_dict



cc_dict = generate_child_chunks(chunks_dict)

,parent_num,child_num,token
count,61.000000,61.000000,61.000000
mean,21.868852,30.000000,262.590164
std,13.562787,17.752934,113.601552
min,1.000000,0.000000,27.000000
25%,9.000000,15.000000,163.000000
50%,23.000000,30.000000,286.000000
75%,34.000000,45.000000,384.000000
max,42.000000,60.000000,384.000000


In [35]:
cc_df = pd.DataFrame(cc_dict)
cc_df.describe()

,parent_num,child_num,token
count,61.000000,61.000000,61.000000
mean,21.868852,30.000000,262.590164
std,13.562787,17.752934,113.601552
min,1.000000,0.000000,27.000000
25%,9.000000,15.000000,163.000000
50%,23.000000,30.000000,286.000000
75%,34.000000,45.000000,384.000000
max,42.000000,60.000000,384.000000
